In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scikit-learn seaborn umap-learn

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

import numpy as np
import matplotlib.pyplot as plt

# Dataset Cleaning

In [ ]:
df_1 = pd.read_csv("../datasets/Oensingen_2018-19.csv")
df_1.columns

In [ ]:
df_1.head(10)

In [ ]:
cols = [
    # Target
    "N2O_flag0_ustar",
    # Timestamp
    "TIMESTAMP",
    # Predictors
    "NEE_f", "GPP_f", "Reco_f", "Rg", "TA", "PREC", "VPD",
    "SWC_0.05", "SWC_0.15", "SWC_0.3",
    "TS_0.05", "TS_0.15", "TS_0.3",
    "harvest", "Norg", "Nmin", "soil"
]

oensingen_18_19 = df_1[cols]

rename_map = {
    "N2O_flag0_ustar": "N2O_Flux",
    "TIMESTAMP": "Timestamp",
    "NEE_f": "NEE",
    "GPP_f": "GPP",
    "Reco_f": "RECO",
    "Rg": "SolarRadiation",
    "TA": "AirTemp",
    "PREC": "Precipitation",
    "SWC_0.05": "SoilWater_5cm",
    "SWC_0.15": "SoilWater_15cm",
    "SWC_0.3": "SoilWater_30cm",
    "TS_0.05": "SoilTemp_5cm",
    "TS_0.15": "SoilTemp_15cm",
    "TS_0.3": "SoilTemp_30cm",
    "harvest": "Mowing",
    "Norg": "FertilizerOrganic",
    "Nmin": "FertilizerMineral",
    "soil": "SoilCultivation",
}

oensingen_1 = oensingen_18_19.rename(columns=rename_map)

# Parse datetime
oensingen_1["Timestamp"] = pd.to_datetime(oensingen_1["Timestamp"], dayfirst=True)
oensingen_1 = oensingen_1.sort_values("Timestamp")
oensingen_1 = oensingen_1.drop_duplicates(subset=["Timestamp"], keep="first")

oensingen_1["year"] = oensingen_1["Timestamp"].dt.year
oensingen_1["month"] = oensingen_1["Timestamp"].dt.month
oensingen_1["hour"] = oensingen_1["Timestamp"].dt.hour
oensingen_1["day"] = oensingen_1["Timestamp"].dt.day

oensingen_1 = oensingen_1.set_index("Timestamp").sort_index()

In [ ]:
oensingen_1.tail(10)

# Further Analysis and Checks

In [ ]:
# ==========================================================
# --- Load Fertilization Info Early ---
# ==========================================================

fert_info = pd.read_csv("../datasets/FertilizationInfo_DataScienceLab/Oensingen_2018-20.csv")
fert_info["date"] = pd.to_datetime(fert_info["date"].astype(str), format="%Y%m%d", errors="coerce")

# Extract only the two ORGANIC fertilization events (exclude mineral as it wasn't N fertilization)
fert_events = fert_info[
    (fert_info["date"].notna()) & 
    (fert_info["N(kg/ha)"].notna()) &
    (fert_info["N(kg/ha)"] > 0) &
    (fert_info["type"] == "organic")  # Only organic fertilizations
].copy()

print(f"Using {len(fert_events)} fertilization events:")
print(fert_events[["date", "N(kg/ha)", "type"]])

In [ ]:
# ==========================================================
# --- Daily Aggregation WITHOUT Auto-Generated DaysSince ---
# ==========================================================

oensingen_1["Date"] = oensingen_1.index.floor("D")

oensingen_1_daily = (
    oensingen_1
    .groupby("Date", dropna=False)
    .agg({
        "N2O_Flux": "mean",
        "NEE": "mean",
        "GPP": "mean",
        "RECO": "mean",
        "SolarRadiation": "mean",
        "AirTemp": "mean",
        "VPD": "mean",
        "SoilWater_5cm": "mean",
        "SoilWater_15cm": "mean",
        "SoilWater_30cm": "mean",
        "SoilTemp_5cm": "mean",
        "SoilTemp_15cm": "mean",
        "SoilTemp_30cm": "mean",
        "Precipitation": "sum",
        # Management flags (keeping for reference, but won't use for DaysSince)
        "Mowing": "max",
        "FertilizerOrganic": "max",
        "FertilizerMineral": "max",
        "SoilCultivation": "max",
    })
    .reset_index()
    .sort_values("Date")
)

oensingen_1_daily = oensingen_1_daily.set_index("Date").asfreq("D")
oensingen_1_daily.index.name = "Date"

In [ ]:
# ==========================================================
# --- Compute Lag Features  ---
# ==========================================================

meteo_daily = [
    "NEE","GPP","RECO",
    "SolarRadiation","AirTemp","VPD",
    "SoilWater_5cm","SoilWater_15cm","SoilWater_30cm",
    "SoilTemp_5cm","SoilTemp_15cm","SoilTemp_30cm",
    "Precipitation"
]

lag_days = [1, 3, 5, 7]
roll_windows = [3, 5, 7]

lagcols = []

# Simple lags
for var in meteo_daily:
    for lag in lag_days:
        lagcols.append(
            oensingen_1_daily[var].shift(lag).rename(f"{var}_lag{lag}d_daily")
        )

# Rolling windows (shifted)
shifted = oensingen_1_daily[meteo_daily].shift(1)

for var in meteo_daily:
    for w in roll_windows:
        lagcols.append(
            shifted[var].rolling(w, min_periods=1).mean().rename(f"{var}_roll{w}d_mean")
        )
        lagcols.append(
            shifted[var].rolling(w, min_periods=1).sum().rename(f"{var}_roll{w}d_sum")
        )

oensingen_1_daily_features = pd.concat(lagcols, axis=1).reset_index()

In [ ]:
# ==========================================================
# --- Merge Lag Features Back to Half-Hourly Data ---
# ==========================================================

oensingen_1 = oensingen_1.reset_index()
oensingen_1["Date"] = oensingen_1["Timestamp"].dt.floor("D")

oensingen_1 = oensingen_1.merge(
    oensingen_1_daily_features,
    how="left",
    on="Date"
)

oensingen_1 = oensingen_1.set_index("Timestamp").sort_index()

In [ ]:
# ==========================================================
# --- MANUAL DaysSince Calculation from CSV Events ---
# ==========================================================

def compute_days_since_fertilization(df, fert_dates, max_days=60):
    """
    Compute days since last fertilization event from a list of fertilization dates.
    
    Parameters:
    - df: DataFrame with DatetimeIndex
    - fert_dates: list of datetime dates when fertilization occurred
    - max_days: cap at this many days
    
    Returns: Series with days since last fertilization
    """
    days_since = pd.Series(index=df.index, dtype=float)
    
    for idx in df.index:
        current_date = idx.floor('D')
        
        # Find all fertilization events before or on current date
        past_events = [d for d in fert_dates if d <= current_date]
        
        if past_events:
            # Days since most recent event
            last_event = max(past_events)
            days = (current_date - last_event).days
            days_since[idx] = min(days, max_days)
        else:
            # No events yet
            days_since[idx] = max_days
    
    return days_since

# Extract fertilization dates from the CSV
fertilization_dates = sorted(fert_events["date"].tolist())

# Compute DaysSince_Fertilization using only CSV events
oensingen_1["DaysSince_Fertilization"] = compute_days_since_fertilization(
    oensingen_1, 
    fertilization_dates, 
    max_days=60
)

# Keep Mowing and SoilCultivation DaysSince (computed from flags)
def days_since_event_halfhourly(series, max_days=30):
    """Days since last management event from binary flags."""
    days = np.full(len(series), max_days, dtype=float)
    last_event_idx = None
    
    for i in range(len(series)):
        if series.iloc[i] == 1:
            last_event_idx = i
            days[i] = 0
        elif last_event_idx is not None:
            days[i] = min(i - last_event_idx, max_days)
        else:
            days[i] = max_days
    
    return days

for event in ["Mowing", "SoilCultivation"]:
    oensingen_1[f"DaysSince_{event}"] = days_since_event_halfhourly(oensingen_1[event])

In [ ]:
# ==========================================================
# --- Filter and Transform ---
# ==========================================================
oensingen_1 = oensingen_1[oensingen_1["N2O_Flux"].notna()]
oensingen_1["N2O_Flux_ln"] = np.where(
    oensingen_1["N2O_Flux"] > 0,
    np.log1p(oensingen_1["N2O_Flux"]),
    0
)

print(f"\nOensingen half-hourly shape: {oensingen_1.shape}")

In [ ]:
# ==========================================================
# --- Daily Aggregation with Manual DaysSince ---
# ==========================================================

oensingen_1["Date"] = oensingen_1.index.floor("D")

oensingen_1_daily = (
    oensingen_1
    .groupby(["Date"], dropna=False)
    .agg({
        # Fluxes and predictors
        "N2O_Flux": "mean",
        "NEE": "mean",
        "GPP": "mean",
        "RECO": "mean",
        "SolarRadiation": "mean",
        "AirTemp": "mean",
        "VPD": "mean",
        "SoilWater_5cm": "mean",
        "SoilWater_15cm": "mean",
        "SoilWater_30cm": "mean",
        "SoilTemp_5cm": "mean",
        "SoilTemp_15cm": "mean",
        "SoilTemp_30cm": "mean",
        "Precipitation": "sum",
        
        # Management events
        "Mowing": "max",
        "SoilCultivation": "max",
        
        # DaysSince features (take minimum of day)
        "DaysSince_Fertilization": "min",
        "DaysSince_Mowing": "min",
        "DaysSince_SoilCultivation": "min",
    })
    .reset_index()
    .sort_values(["Date"])
)

# Set index
oensingen_1_daily = oensingen_1_daily.set_index("Date").sort_index()
oensingen_1_daily.index.name = "Date"

In [ ]:
# ==========================================================
# --- Add Daily Lag Features ---
# ==========================================================

newcols = []

for var in meteo_daily:
    for lag in lag_days:
        newcols.append(
            oensingen_1_daily[var].shift(lag).rename(f"{var}_lag{lag}d")
        )

shifted_daily = oensingen_1_daily[meteo_daily].shift(1)

for var in meteo_daily:
    for w in roll_windows:
        newcols.append(
            shifted_daily[var].rolling(w, min_periods=1).mean().rename(f"{var}_roll{w}d_mean")
        )
        newcols.append(
            shifted_daily[var].rolling(w, min_periods=1).sum().rename(f"{var}_roll{w}d_sum")
        )

oensingen_1_daily = pd.concat([oensingen_1_daily] + newcols, axis=1).reset_index()

In [ ]:
# ==========================================================
# --- Merge Fertilizer Amounts from CSV ---
# ==========================================================

oensingen_1_daily["Date"] = pd.to_datetime(oensingen_1_daily["Date"], errors="coerce")

oensingen_1_daily = oensingen_1_daily.merge(
    fert_info[["date", "N(kg/ha)", "type"]],
    how="left",
    left_on="Date",
    right_on="date"
)

oensingen_1_daily.drop(columns=["date"], inplace=True)
oensingen_1_daily.rename(columns={"N(kg/ha)": "Fertilizer_N_kg_ha"}, inplace=True)

# Add temporal features
oensingen_1_daily["year"] = oensingen_1_daily["Date"].dt.year
oensingen_1_daily["month"] = oensingen_1_daily["Date"].dt.month

# Filter and transform
oensingen_1_daily = oensingen_1_daily[oensingen_1_daily["N2O_Flux"].notna()]
oensingen_1_daily["N2O_Flux_ln"] = np.where(
    oensingen_1_daily["N2O_Flux"] > 0,
    np.log1p(oensingen_1_daily["N2O_Flux"]),
    0
)

oensingen_1_daily = oensingen_1_daily.set_index("Date").sort_index()

print(f"\nOensingen daily shape: {oensingen_1_daily.shape}")

In [ ]:
# ==========================================================
# --- Exponential Decay Fertilizer Stock ---
# ==========================================================

def add_exponential_decay_single(df, dose_col="Fertilizer_N_kg_ha",
                                 half_lives=(3, 7, 14)):
    """
    Calendar-aware exponential decay for datasets with missing days.
    """
    df = df.sort_index().copy()
    dose = df[dose_col].fillna(0).to_numpy(float)
    
    dates = df.index.to_series()
    delta_days = dates.diff().dt.total_seconds().div(86400).fillna(0).to_numpy()
    
    for hl in half_lives:
        stock = np.zeros_like(dose)
        acc = 0.0
        
        for i, x in enumerate(dose):
            if i == 0:
                acc = x
            else:
                decay = 0.5 ** (delta_days[i] / hl)
                acc = x + decay * acc
            stock[i] = acc
        
        df[f"{dose_col}_expHL{hl}d"] = stock
    
    return df

if "Fertilizer_N_kg_ha" in oensingen_1_daily.columns:
    oensingen_1_daily = add_exponential_decay_single(oensingen_1_daily)
    print("\nExponential decay features added.")
else:
    print("\nNo Fertilizer_N_kg_ha column — skipping exponential decay.")

In [ ]:
# ==========================================================
# --- Verification ---
# ==========================================================

print("\n" + "="*60)
print("VERIFICATION: Fertilization Events and DaysSince")
print("="*60)

# Show days around fertilization events
for fert_date in fertilization_dates:
    print(f"\n--- Around fertilization on {fert_date.date()} ---")
    window = oensingen_1_daily[
        (oensingen_1_daily.index >= fert_date - pd.Timedelta(days=2)) &
        (oensingen_1_daily.index <= fert_date + pd.Timedelta(days=5))
    ][["DaysSince_Fertilization", "Fertilizer_N_kg_ha"]]
    print(window)

print("\n" + "="*60)
print("Summary of DaysSince_Fertilization:")
print(oensingen_1_daily["DaysSince_Fertilization"].describe())
print("="*60)

In [ ]:
oensingen_1.to_csv("../datasets/Oensingen_2018-19_clean_newlag.csv")
oensingen_1_daily.to_csv("../datasets/Oensingen_Daily_2018-19_clean_newlag.csv")

In [ ]:
oensingen_1_daily[oensingen_1_daily["Fertilizer_N_kg_ha"].notna()]

# Analysis Plots

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute correlation matrix (Pearson by default)
corr = oensingen_1.corr(numeric_only=True)

plt.figure(figsize=(32,30))
sns.heatmap(
    corr, 
    annot=True, fmt=".2f", cmap="coolwarm",
    cbar_kws={'label': 'Correlation'}
)
plt.title("Correlation matrix (including target FN2O)")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

def plot_time_series(df, vars_to_plot):
    """
    Plot time series of selected variables with real time gaps shown on the x-axis.
    Automatically formats time labels and adds axis labels.
    """
    df = df.copy().sort_index()

    # --- Ensure datetime index ---
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # --- Layout ---
    n_cols = 3
    n_rows = int(len(vars_to_plot) / n_cols) + (len(vars_to_plot) % n_cols > 0)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 12), sharex=True)
    axes = axes.flatten()

    # --- Plot each variable ---
    for i, var in enumerate(vars_to_plot):
        ax = axes[i]
        ax.plot(df.index, df[var], lw=1)
        ax.set_title(var, fontsize=10)
        ax.set_ylabel(var)
        ax.grid(True, alpha=0.3)

        # Format the x-axis as dates
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.tick_params(axis='x', rotation=45)

    # --- Remove unused axes if any ---
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # --- Common labels and formatting ---
    fig.suptitle("Time Series", fontsize=14)
    fig.text(0.5, 0.04, "Date", ha='center', fontsize=12)
    fig.tight_layout(rect=[0, 0.05, 1, 0.97])
    plt.show()

In [ ]:
# Variables to plot for half-hourly data (still has the original flags)
vars_to_plot = [
    "N2O_Flux", "NEE", "GPP", "RECO",
    "SolarRadiation", "AirTemp", "Precipitation",
    "VPD", "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "Mowing", "SoilCultivation"  # Removed FertilizerOrganic and FertilizerMineral
]

# Variables to plot for daily data (has CSV-based fertilization)
vars_to_plot_w_fert = [
    "N2O_Flux", "NEE", "GPP", "RECO",
    "SolarRadiation", "AirTemp", "Precipitation",
    "VPD", "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "Mowing", "SoilCultivation",  # Removed FertilizerOrganic and FertilizerMineral
    "Fertilizer_N_kg_ha",  # From CSV
    "Fertilizer_N_kg_ha_expHL3d", "Fertilizer_N_kg_ha_expHL7d", "Fertilizer_N_kg_ha_expHL14d"
]

# Daily averages
print("Raw Data")
plot_time_series(oensingen_1, vars_to_plot)
print("Daily Freq")
plot_time_series(oensingen_1_daily, vars_to_plot_w_fert)

In [ ]:
# count observations per hour
hourly_counts = oensingen_1["hour"].value_counts().sort_index()

# plot
plt.figure(figsize=(8,4))
hourly_counts.plot(kind="bar")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Observations")
plt.title("Observation Frequency by Hour of Day")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# count per month
monthly_counts = oensingen_1["month"].value_counts().sort_index()

# plot
plt.figure(figsize=(8,4))
monthly_counts.plot(kind="bar", color="tab:green")
plt.xlabel("Month")
plt.ylabel("Number of Observations")
plt.title("Observation Frequency by Month")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# group by year × hour
hour_year_counts = (
    oensingen_1.groupby(["year", "hour"])
    .size()
    .unstack(fill_value=0)
)

# normalize by total per year (to compare proportions)
hour_year_norm = hour_year_counts.div(hour_year_counts.sum(axis=1), axis=0)

# plot as heatmap
plt.figure(figsize=(12,6))
sns.heatmap(hour_year_norm, cmap="viridis", cbar_kws={"label": "Fraction of daily observations"})
plt.xlabel("Hour of Day")
plt.ylabel("Year")
plt.title("Distribution of Observation Hours Across Years")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# compute median N₂O flux per year × hour
median_flux = (
    oensingen_1.groupby(["year", "month"])["N2O_Flux"]
    .median()
    .unstack(fill_value=np.nan)
)

# plot heatmap
plt.figure(figsize=(12,6))
sns.heatmap(
    median_flux,
    cmap="RdYlBu_r",
    cbar_kws={"label": "Median N₂O Flux"},
)
plt.xlabel("Month")
plt.ylabel("Year")
plt.title("Median N₂O Flux by Month and Year")
plt.tight_layout()
plt.show()

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# define bin edges for both temperatures
bins_5cm = np.arange(oensingen_1["SoilTemp_5cm"].min(), oensingen_1["SoilTemp_5cm"].max()+1, 1)
bins_15cm = np.arange(oensingen_1["SoilTemp_15cm"].min(), oensingen_1["SoilTemp_15cm"].max()+1, 1)

# create binned categories
oensingen_1["T5_bin"] = pd.cut(oensingen_1["SoilTemp_5cm"], bins=bins_5cm)
oensingen_1["T15_bin"] = pd.cut(oensingen_1["SoilTemp_15cm"], bins=bins_15cm)

# compute median flux per 2D bin
median_flux_binned = (
    oensingen_1.groupby(["T5_bin", "T15_bin"])["N2O_Flux"]
    .median()
    .unstack(fill_value=np.nan)
)

# plot heatmap
plt.figure(figsize=(10,8))
sns.heatmap(
    median_flux_binned,
    cmap="RdYlBu_r",
    cbar_kws={"label": "Median N₂O Flux"},
)
plt.xlabel("Soil Temperature 15 cm (°C)")
plt.ylabel("Soil Temperature 5 cm (°C)")
plt.title("Median N₂O Flux by Binned Soil Temperatures (5 cm vs 15 cm)")
plt.tight_layout()
plt.show()

In [ ]:
hourly_mean = oensingen_1.groupby("hour")["N2O_Flux"].mean()
hourly_std  = oensingen_1.groupby("hour")["N2O_Flux"].std()

plt.figure(figsize=(8, 5))
plt.errorbar(hourly_mean.index, hourly_mean, yerr=hourly_std, fmt="-o", capsize=3)
plt.xlabel("Hour of Day")
plt.ylabel("Mean N₂O Flux")
plt.title("Average Diurnal Cycle of N₂O Flux")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

monthly_mean = oensingen_1.groupby("month")["N2O_Flux"].mean()
monthly_std  = oensingen_1.groupby("month")["N2O_Flux"].std()

plt.figure(figsize=(8, 5))
plt.errorbar(monthly_mean.index, monthly_mean, yerr=monthly_std, fmt="-o", capsize=3)
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Mean N₂O Flux")
plt.title("Seasonal Cycle of N₂O Flux")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Base setup ---
oensingen_1_time_index = oensingen_1.copy()

predictors = [
    "NEE", "GPP", "RECO",
    "SolarRadiation", "AirTemp", "Precipitation", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
]

target = "N2O_Flux"

# --- Parameters ---
window_hours = 24        # window size for averaging (past 24h)
step_hours   = 6        # step between lags
max_hours    = 7 * 24    # look back 1 week (you can extend to 5 weeks)
offsets      = range(0, -max_hours - step_hours, -step_hours)  # only past (0, -24, -48, ...)

# --- Rolling mean of predictors (24h backward window) ---
window = f"{window_hours}h"
rolling_means = oensingen_1_time_index[predictors].rolling(window=window, closed="left").mean()

# --- Compute lag correlations (Spearman) ---
lag_corrs = {}

for var in predictors:
    corrs = []
    for offset in offsets:
        shifted = rolling_means[var].shift(freq=pd.Timedelta(hours=offset))
        aligned = oensingen_1_time_index[[target]].join(shifted.rename("past_mean")).dropna()
        if len(aligned) > 2:
            rho, _ = spearmanr(aligned[target], aligned["past_mean"])
            corrs.append(rho)
        else:
            corrs.append(np.nan)
    lag_corrs[var] = (list(offsets), corrs)

# --- Plot results ---
plt.figure(figsize=(12, 8))
for var, (offsets, corrs) in lag_corrs.items():
    plt.plot(offsets, corrs, label=var)

plt.axvline(0, color="k", linestyle="--", lw=1)
plt.xlabel("Hours before N₂O flux measurement (negative = further in past)")
plt.ylabel("Spearman ρ (past mean condition vs N₂O Flux)")
plt.title("Time-Lagged Spearman Correlation of Past Mean Conditions vs N₂O Flux")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr

def plot_lag_correlation(df, col, target="N2O_Flux", max_lag=60, resample_daily=True):
    """
    Compute and plot the Spearman correlation between a predictor and target variable
    over increasing day lags (calendar-based).

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with a DatetimeIndex.
    col : str
        Column name of the predictor variable (e.g. "Precipitation").
    target : str, optional
        Column name of the target variable, by default "N2O_Flux".
    max_lag : int, optional
        Maximum lag in days, by default 60.
    resample_daily : bool, optional
        If True, resample to daily mean (useful if data is sub-daily).

    Returns
    -------
    pd.DataFrame
        DataFrame with 'lag' and 'spearman_r' for each lag.
    """

    # --- Ensure datetime index ---
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # --- Optional daily resampling ---
    if resample_daily:
        df = df.select_dtypes(include=[np.number]).resample("D").mean()

    # --- Filter for valid positive flux values ---
    df = df[df[target] >= 0][[col, target]].dropna(subset=[target])

    # --- Compute lag correlations ---
    lags = np.arange(0, max_lag + 1)
    corrs = []

    for lag in lags:
        shifted = df[col].shift(freq=pd.to_timedelta(lag, unit="D"))
        aligned = pd.concat([df[target], shifted], axis=1, join="inner").dropna()

        if aligned.empty:
            corrs.append(np.nan)
        else:
            r, _ = spearmanr(aligned[target], aligned[col])
            corrs.append(r)

    # --- Store results ---
    result = pd.DataFrame({"lag_days": lags, "spearman_r": corrs})

    # --- Plot ---
    plt.figure(figsize=(8, 4))
    plt.plot(result["lag_days"], result["spearman_r"], marker="o", color="tab:blue")
    plt.axhline(0, color="gray", lw=1)
    plt.xlabel(f"Lag (days after {col})")
    plt.ylabel("Spearman ρ")
    plt.title(f"Spearman correlation between {col} and future {target}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- Report peak correlation ---
    best_lag = result["lag_days"].iloc[np.nanargmax(result["spearman_r"])]
    best_r = np.nanmax(result["spearman_r"])
    print(f"Peak Spearman correlation at lag = {best_lag} days (ρ = {best_r:.3f})")

plot_lag_correlation(oensingen_1, col="Precipitation", max_lag=60)
plot_lag_correlation(oensingen_1, col="AirTemp", max_lag=45)
plot_lag_correlation(oensingen_1, col="VPD", max_lag=30)

In [ ]:
def cross_correlation_single(df, predictor_vars, target="N2O_Flux", max_lag=60):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"])
            df = df.set_index("Date")
        else:
            raise ValueError("DataFrame must have a DatetimeIndex or 'Date' column")

    df = df[df[target] >= 0].dropna(subset=[target])
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    predictor_vars = [v for v in predictor_vars if v in numeric_cols]
    lags = np.arange(-max_lag, max_lag + 1)
    
    # Sort by index
    df = df.sort_index()
    corrs_dict = {}

    # Compute correlations for each predictor
    for var in predictor_vars:
        corrs = []
        for lag in lags:
            shifted = df[var].shift(freq=pd.to_timedelta(lag, unit="D"))
            aligned = pd.concat([df[target], shifted], axis=1, join="inner").dropna()
            rho = np.nan if aligned.empty else spearmanr(aligned[target], aligned[var])[0]
            corrs.append(rho)
        corrs_dict[var] = corrs

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for var, corrs in corrs_dict.items():
        ax.plot(lags, corrs, lw=1.2, label=var)

    ax.axhline(0, color="gray", lw=1)
    ax.axvline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel("Lag (days)")
    ax.set_ylabel("Spearman ρ")
    ax.set_title("Cross-correlation between predictors and N₂O flux", fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()

    # --- Summary ---
    summary_rows = []
    for var, corrs in corrs_dict.items():
        best_idx = np.nanargmax(np.abs(corrs))
        summary_rows.append({
            "Variable": var,
            "Best lag (days)": int(lags[best_idx]),
            "Max corr (ρ)": corrs[best_idx],
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df = (
        summary_df
        .sort_values(
            by="Max corr (ρ)",
            ascending=False,
            key=lambda col: np.abs(col) if col.name == "Max corr (ρ)" else col
        )
        .round(3)
    )

    return summary_df

predictors = [
    "Precipitation", "AirTemp", "VPD", "GPP", "RECO",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm"
]

summary = cross_correlation_single(oensingen_1_daily, predictor_vars=predictors, max_lag=60)
print(summary)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Identify fertilization events ---
fert_events = oensingen_1_daily[
    (oensingen_1_daily["FertilizerOrganic"] == 1) |
    (oensingen_1_daily["FertilizerMineral"] == 1)
].copy()

# Date is already the index, so we'll work with that
print(f"Found {len(fert_events)} fertilization events")

# --- Extract N₂O fluxes for 14 days after each event ---
window_days = 14
records = []

for event_date, event in fert_events.iterrows():
    # event_date is already a datetime since it's the index
    subset = oensingen_1_daily[
        (oensingen_1_daily.index >= event_date) &
        (oensingen_1_daily.index <= event_date + pd.Timedelta(days=window_days))
    ].copy()
    
    subset["days_since_fert"] = (subset.index - event_date).days
    subset["event_date"] = event_date
    subset["fert_type"] = (
        "Organic" if event["FertilizerOrganic"] == 1 else "Mineral"
    )
    records.append(subset)

fert_windows = pd.concat(records, ignore_index=False)

# --- Plot setup (1x2 grid) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("N₂O Flux response after fertilization (14-day window)", fontsize=14)

# --- Panel 1: All individual events ---
for event_date, group in fert_windows.groupby("event_date"):
    axes[0].plot(group["days_since_fert"], group["N2O_Flux"], marker="o", alpha=0.6)
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("All fertilization events")
axes[0].set_xlabel("Days since fertilization")
axes[0].set_ylabel("N₂O Flux")

# --- Panel 2: Average by fertilizer type ---
for fert_type, group in fert_windows.groupby("fert_type"):
    mean_curve = group.groupby("days_since_fert")["N2O_Flux"].mean()
    axes[1].plot(mean_curve.index, mean_curve.values, marker="o", label=fert_type, linewidth=2)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Average by fertilizer type")
axes[1].set_xlabel("Days since fertilization")
axes[1].set_ylabel("N₂O Flux")
axes[1].legend(title="Fertilizer Type")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# ==========================================================
# 1️⃣ Helper: extract N₂O flux windows after each management event
# ==========================================================
def extract_event_windows(df_daily, event_col, window_days=14):
    """
    Extract N₂O flux time windows following a management event.

    Args:
        df_daily : daily dataframe (must have Date as index, N2O_Flux, and event_col)
        event_col : column name for event (e.g., "Mowing")
        window_days : number of days after event to include
    """
    records = []
    events = df_daily[df_daily[event_col] == 1].copy()
    if events.empty:
        return pd.DataFrame()

    for event_date, event in events.iterrows():
        subset = df_daily[
            (df_daily.index >= event_date) &
            (df_daily.index <= event_date + pd.Timedelta(days=window_days))
        ].copy()

        subset["days_since_event"] = (subset.index - event_date).days
        subset["event_date"] = event_date
        subset["event_type"] = event_col  # keep event label
        records.append(subset)

    return pd.concat(records, ignore_index=False)


# ==========================================================
# 2️⃣ Create windows for each management type
# ==========================================================
mow_windows   = extract_event_windows(oensingen_1_daily, "Mowing")
cult_windows  = extract_event_windows(oensingen_1_daily, "SoilCultivation")
fert_windows  = extract_event_windows(oensingen_1_daily, "FertilizerOrganic")
fert_windowsM = extract_event_windows(oensingen_1_daily, "FertilizerMineral")

# merge organic + mineral
fert_windows["event_type"] = "Organic"
fert_windowsM["event_type"] = "Mineral"
fert_windows = pd.concat([fert_windows, fert_windowsM], ignore_index=False)

# Add fertilizer amount (if available)
if "Fertilizer_N_kg_ha" not in fert_windows.columns:
    fert_windows["Fertilizer_N_kg_ha"] = 0.0

# ==========================================================
# 3️⃣ Fertilization plots
# ==========================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle("N₂O Flux after fertilization (14-day window)", fontsize=14)

# normalize fertilizer N for color mapping
norm = mcolors.Normalize(
    vmin=fert_windows["Fertilizer_N_kg_ha"].min(),
    vmax=fert_windows["Fertilizer_N_kg_ha"].max()
)
cmap = cm.viridis


def plot_fert_events(ax, df, title):
    if df.empty:
        ax.set_title(f"{title}\n(no events)", fontsize=10)
        return

    n_total = df["event_date"].nunique()

    for event_date, group in df.groupby("event_date"):
        n_val = group["Fertilizer_N_kg_ha"].iloc[0]
        fert_type = group["event_type"].iloc[0]
        color = cmap(norm(n_val))
        linestyle = "-" if "Organic" in fert_type else ":"
        ax.plot(
            group["days_since_event"], group["N2O_Flux"],
            color=color, linestyle=linestyle, marker="o", alpha=0.9
        )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{title}\n(n = {n_total})", fontsize=11)
    ax.set_xlabel("Days since event")
    ax.set_ylabel("N₂O Flux")
    ax.grid(True, alpha=0.3)


# (a) All events
plot_fert_events(axes[0], fert_windows, "All events")

# (b) Average by fertilizer type
ax = axes[1]
for event_type, group in fert_windows.groupby("event_type"):
    mean_curve = group.groupby("days_since_event")["N2O_Flux"].mean()
    linestyle = "-" if "Organic" in event_type else ":"
    ax.plot(mean_curve.index, mean_curve.values,
            marker="o", label=event_type, linestyle=linestyle)
n_total = fert_windows["event_date"].nunique()
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_title(f"Average by fertilizer type\n(n = {n_total})", fontsize=11)
ax.set_xlabel("Days since event")
ax.legend(title="Fertilizer type", fontsize=8)
ax.grid(True, alpha=0.3)

# colorbar
cbar_ax = fig.add_axes([0.25, 0.04, 0.5, 0.02])
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Fertilizer N (kg ha⁻¹)")

plt.tight_layout(rect=[0, 0.07, 1, 0.96])
plt.show()

# ==========================================================
# 4️⃣ Management event plots
# ==========================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle("N₂O Flux after management events (14-day window)", fontsize=14)

management_panels = [
    (mow_windows, "Mowing", "green"),
    (cult_windows, "Soil Cultivation", "brown"),
    (fert_windows, "Fertilization", "purple")
]

for ax, (df, label, color) in zip(axes, management_panels):
    if df.empty:
        ax.set_title(f"{label}\n(no events)", fontsize=10)
        continue

    n_total = df["event_date"].nunique()

    mean_curve = df.groupby("days_since_event")["N2O_Flux"].mean()
    ax.plot(
        mean_curve.index, mean_curve.values,
        marker="o", color=color, linestyle="-"
    )

    ax.axvline(0, color="black", linestyle="--")
    ax.set_title(f"{label}\n(n = {n_total})", fontsize=11)
    ax.set_xlabel("Days since event")
    ax.set_ylabel("N₂O Flux")
    ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# PCA plots

In [ ]:
oensingen_1_clean = oensingen_1.dropna()
oensingen_1_clean = oensingen_1_clean.sort_values(
    by="N2O_Flux",
    ascending=True  # ascending puts zeros first, descending puts them last
)

X_1 = oensingen_1_clean.drop(columns=["N2O_Flux", "Timestamp", "time_diff", "Date"])
y_1 = oensingen_1_clean["N2O_Flux"]

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Standardize predictors
X_scaled_1 = StandardScaler().fit_transform(X_1)

# Run PCA (2 components for visualization)
pca_1 = PCA(n_components=2)
X_pca_1 = pca_1.fit_transform(X_scaled_1)

# Create DataFrame for plotting
pca_df_1 = pd.DataFrame(X_pca_1, columns=['PC1', 'PC2'])
pca_df_1["FN2O"] = y_1.values

# Scatter plot
plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df_1, x="PC1", y="PC2", hue="FN2O", palette="coolwarm")
plt.title("PCA projection (colored by N2O flux)")
plt.xlabel(f"PC1 ({pca_1.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca_1.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="FN2O flux", loc="best")
plt.show()

In [ ]:
loadings = pd.DataFrame(
    pca_1.components_.T,
    index=X_1.columns,
    columns=["PC1", "PC2"]
)
print(loadings.sort_values("PC1", ascending=False))

In [ ]:
import matplotlib.cm as cm

# --- transformation ---
oensingen_1_clean["N2O_Flux_ln"] = np.where(
    oensingen_1_clean["N2O_Flux"] > 0,
    np.log1p(oensingen_1_clean["N2O_Flux"]),
    0
)

# =============================================
# 1️⃣ Define variables and plotting function
# =============================================
base_vars = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO", "Mowing", "FertilizerMineral", "SoilCultivation"
]

def plot_pca(df, title):
    # Drop rows with missing values in required columns
    df_clean = df.dropna(subset=base_vars + ["N2O_Flux_ln"]).copy()
    
    # Standardize variables
    X_scaled = StandardScaler().fit_transform(df_clean[base_vars])
    
    # Run PCA
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)
    df_clean["PC1"] = pcs[:, 0]
    df_clean["PC2"] = pcs[:, 1]

    # Sort by N2O_Flux (so larger appear on top)
    df_clean = df_clean.sort_values("N2O_Flux_ln", ascending=True)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        df_clean["PC1"], df_clean["PC2"],
        c=df_clean["N2O_Flux_ln"],
        cmap=cm.viridis,
        s=40, alpha=0.8, edgecolor="none"
    )

    # Labels and aesthetics
    ax.set_title(f"{title}\nPCA of environmental variables", fontsize=13)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("ln(N₂O Flux)")
    plt.tight_layout()
    plt.show()

# =============================================
# 2️⃣ Run for oensingen 2018-19
# =============================================
plot_pca(oensingen_1_clean, "Oensingen 2018-19")

# UMAP plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.cm as cm

# =============================================
# 1️⃣ Define variables and plotting function
# =============================================
base_vars = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO", "Mowing", "FertilizerMineral", "SoilCultivation"
]


def plot_umap(df, title, n_neighbors=50, min_dist=0.5, random_state=42):
    # Drop rows with missing values
    df_clean = df.dropna(subset=base_vars + ["N2O_Flux"]).copy()

    # Standardize
    X_scaled = StandardScaler().fit_transform(df_clean[base_vars])

    # Run UMAP
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state
    )
    embedding = reducer.fit_transform(X_scaled)
    df_clean["UMAP1"] = embedding[:, 0]
    df_clean["UMAP2"] = embedding[:, 1]

    # Sort by N2O_Flux so high values are plotted last (on top)
    df_clean = df_clean.sort_values("N2O_Flux", ascending=True)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        df_clean["UMAP1"], df_clean["UMAP2"],
        c=df_clean["N2O_Flux"],
        cmap=cm.viridis,
        s=20, alpha=0.6, edgecolor="none"
    )

    ax.set_title(f"{title}\nUMAP of environmental variables", fontsize=13)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("N₂O Flux")
    plt.tight_layout()
    plt.show()

# =============================================
# 2️⃣ Run for Oensingen 2018-19
# =============================================
plot_umap(oensingen_1_clean, "Oensingen 2018-19")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.cm as cm

# =============================================
# 1️⃣ Define variables and plotting function
# =============================================
base_vars = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO", "Mowing", "FertilizerMineral", "SoilCultivation"
]


def plot_umap(df, title, n_neighbors=50, min_dist=0.5, random_state=42):
    # Drop rows with missing values
    df_clean = df.dropna(subset=base_vars + ["N2O_Flux_ln"]).copy()

    # Standardize
    X_scaled = StandardScaler().fit_transform(df_clean[base_vars])

    # Run UMAP
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state
    )
    embedding = reducer.fit_transform(X_scaled)
    df_clean["UMAP1"] = embedding[:, 0]
    df_clean["UMAP2"] = embedding[:, 1]

    # Sort by N2O_Flux so high values are plotted last (on top)
    df_clean = df_clean.sort_values("N2O_Flux_ln", ascending=True)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        df_clean["UMAP1"], df_clean["UMAP2"],
        c=df_clean["N2O_Flux_ln"],
        cmap=cm.viridis,
        s=20, alpha=0.6, edgecolor="none"
    )

    ax.set_title(f"{title}\nUMAP of environmental variables", fontsize=13)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("ln(N₂O Flux)")
    plt.tight_layout()
    plt.show()

# =============================================
# 2️⃣ Run for Oensingen 2018-19
# =============================================
plot_umap(oensingen_1_clean, "Oensingen 2018-19")

# t-SNE plots

In [ ]:
from sklearn.preprocessing import StandardScaler

# Drop or fill NaNs
X_1 = X_1.dropna()
y_1 = y_1.loc[X_1.index]

# Scale predictors
X_scaled_1 = StandardScaler().fit_transform(X_1)

In [ ]:
from sklearn.manifold import TSNE

tsne_1 = TSNE(n_components=2, perplexity=90, random_state=42)
X_tsne_1 = tsne_1.fit_transform(X_scaled_1)

tsne_df_1 = pd.DataFrame(X_tsne_1, columns=["tSNE1", "tSNE2"])
tsne_df_1["FN2O"] = y_1.values

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=tsne_df_1, x="tSNE1", y="tSNE2",
    hue="FN2O", palette="coolwarm", s=40
)
plt.title("t-SNE embedding coloured by N₂O flux")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================
# 1️⃣ Define variables and plotting function
# =============================================
base_vars = [
    "Precipitation", "SolarRadiation", "AirTemp", "VPD",
    "SoilWater_5cm", "SoilWater_15cm", "SoilWater_30cm",
    "SoilTemp_5cm", "SoilTemp_15cm", "SoilTemp_30cm",
    "NEE", "GPP", "RECO", "Mowing", "FertilizerMineral", "SoilCultivation"
]

def plot_tsne(df, title, perplexity=100, learning_rate=200, random_state=42):
    # Drop missing values
    df_clean = df.dropna(subset=base_vars + ["N2O_Flux_ln"]).copy()

    # Standardize features
    X_scaled = StandardScaler().fit_transform(df_clean[base_vars])

    # Run t-SNE
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate=learning_rate,
        init="pca",
        random_state=random_state
    )
    tsne_results = tsne.fit_transform(X_scaled)

    df_clean["tSNE1"] = tsne_results[:, 0]
    df_clean["tSNE2"] = tsne_results[:, 1]

    # Sort by N2O_Flux for plotting order
    df_clean = df_clean.sort_values("N2O_Flux_ln", ascending=True)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        df_clean["tSNE1"], df_clean["tSNE2"],
        c=df_clean["N2O_Flux_ln"],
        cmap=cm.viridis,
        s=40, alpha=0.8, edgecolor="none"
    )

    ax.set_title(f"{title}\nt-SNE of environmental variables", fontsize=13)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("ln(N₂O Flux)")
    plt.tight_layout()
    plt.show()

# =============================================
# 2️⃣ Run for Oensingen 2018-19
# =============================================
plot_tsne(oensingen_1_clean, "Oensingen 2018-19")